# Direction 2 · Phase C · Step 6b: Peak-Neutral Pre-Layer4 Repair & Layer4.conv2 LoRA (Kaggle GPU)

**Settings:** Enable **GPU T4 x1** and **Internet ON**.

**Zero manual uploads required:**
- Clones `spurious_correlation_mechanisms` from GitHub.
- Auto-downloads the 5 saved `standard_erm` ResNet-50 `ckpt_last.pt` backbones, Waterbirds images (~490 MB), and Caltech `CUB_200_2011` pixel masks (~39 MB, used **strictly for post-hoc counterfactual & gate-alignment evaluation, never during training**).
- Tests all 3 fixes to Step 6 + the Step 5 Panel B `layer4.{0,1,2}.conv2` rank-8 LoRA control on cleaned 5-fold validation CV (`split == 1`, zero test access):
  1. `affine_dfr_linear` (reference baseline: `bn_all` + pretrained linear head)
  2. `pre_l4_gate_bn_ce` (peak-normalized neutral-mean `layer3.5` gate + `bn_all` + linear, CE only)
  3. `pre_l4_repair_w005` (peak-neutral gate + `bn_all` + linear + gentle `drift_w=0.05`)
  4. `pre_l4_repair_w015` (peak-neutral gate + `bn_all` + linear + moderate `drift_w=0.15`)
  5. `l4_conv2_lora_bn` (rank-8 LoRA on `layer4.{0,1,2}.conv2` only + `bn_all` + linear, CE only)
  6. `l4_conv2_lora_repair_w005` (peak-neutral gate + `layer4.conv2` LoRA + `bn_all` + linear + `drift_w=0.05`)
- Outputs `/kaggle/working/rep_repair_phase_c_step6b.zip`.

In [ ]:
import os, shutil, subprocess, sys, zipfile
from pathlib import Path
import torch

REPO_URL = 'https://github.com/chautanphat/waterbirds-shortcut-mitigation.git'
BRANCH = 'spurious_correlation_mechanisms'
CODE_DIR = Path('/kaggle/working/repo')
WORK_DIR = Path('/kaggle/working/phase_c_assets')
OUT_DIR = Path('/kaggle/working/rep_repair_phase_c_step6b')

print('CUDA available:', torch.cuda.is_available(), '| Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
if CODE_DIR.exists():
    shutil.rmtree(CODE_DIR)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(CODE_DIR)], check=True)
os.chdir(CODE_DIR)
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

# Pre-flight unit tests.
subprocess.run([sys.executable, '-B', '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_rep_repair_phase_c_step6.py', '-v'], check=True)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
cmd = [
    sys.executable, '-B', '-m', 'scripts.analysis.rep_repair_phase_c_step6b',
    '--work-dir', str(WORK_DIR),
    '--output', str(OUT_DIR),
    '--device', device,
    '--workers', '2',
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
from IPython.display import Image, Markdown, display

display(Markdown((OUT_DIR / 'README.md').read_text(encoding='utf-8')))
display(Image(filename=str(OUT_DIR / 'step6b_ablation.png')))
display(Image(filename=str(OUT_DIR / 'gate_maps_step6b.png')))

zip_path = Path('/kaggle/working/rep_repair_phase_c_step6b.zip')
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(OUT_DIR.rglob('*')):
        if f.is_file():
            zf.write(f, arcname=f.relative_to(OUT_DIR))
print(f'Saved results bundle: {zip_path} ({zip_path.stat().st_size / 1e6:.2f} MB)')